# RNN Encoder-Decoder with Additive Attention (lab 9 AI)

## Aakrit Pahadi(ACE080BCT002)

Objective:
To design, implement, and train a sequence-to-sequence Recurrent Neural Network (RNN) model using an encoder-decoder architecture with additive attention for translating short French sentences into English.

## Theory

The **Sequence-to-Sequence (Seq2Seq)** or **Encoder-Decoder** architecture is a neural network model designed to convert one sequence of data into another. It is commonly applied in various Natural Language Processing (NLP) tasks, including machine translation, text summarization, speech recognition, chatbot systems, and question-answering applications.

A standard Seq2Seq model consists of two main components:

1. **Encoder**
2. **Decoder**

The encoder receives the input sequence one token at a time and generates hidden representations that capture the meaning of the input. In the traditional Seq2Seq model, only the encoder's final hidden state is passed to the decoder as a context vector. Since this single vector represents the entire input sentence, important information may be lost when processing long or complex sequences.

To overcome this limitation, the **Bahdanau (Additive) Attention** mechanism is introduced. Rather than depending only on the final encoder state, attention enables the decoder to consider all encoder hidden states during each decoding step. It calculates attention scores for every input token and combines them into a weighted context vector, allowing the model to focus on the most relevant parts of the input while generating each output word.

## Working Principle

The attention-based Seq2Seq translation model operates through the following steps:

1. The input sentence is divided into individual tokens.
2. Each token is converted into its corresponding numerical representation, such as an index or embedding vector.
3. The encoder processes the entire input sequence and produces hidden states for every input token.
4. The decoder begins generating the output sequence using the **Start-of-Sequence (SOS)** token.
5. During each decoding step, the Bahdanau attention mechanism computes attention scores between the decoder's current hidden state and all encoder hidden states.
6. These attention scores are converted into weights, which are used to create a context vector representing the most relevant input information.
7. The decoder combines the context vector with the current input to predict the next word in the target language.
8. The decoding process continues until the **End-of-Sequence (EOS)** token is produced or the predefined maximum sequence length is reached.

In [44]:
# requirements
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [45]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [46]:
# Turn a Unicode string to plain ASCII, thanks to
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [47]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [48]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [49]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [50]:
PATH = 'data.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes rusees', 'you re crafty']


15

### Encoder

The encoder is responsible for processing the input sentence and converting it into meaningful hidden representations. In this implementation, the encoder consists of an embedding layer followed by a Recurrent Neural Network (RNN).

For each input token:

- The input token is transformed into a dense embedding vector.
- The embedding is passed through the RNN together with the previous hidden state.
- The RNN generates a new hidden state, which is saved as part of the encoder outputs.

In an attention-based Seq2Seq model, every encoder hidden state is preserved instead of using only the final one. While the last hidden state is used to initialize the decoder, the complete sequence of encoder outputs allows the attention mechanism to focus on the most relevant parts of the input during each decoding step.

Mathematically,

\[
h_t = f(x_t, h_{t-1})
\]

where:

- \(x_t\) = embedding vector of the input token at time step \(t\)
- \(h_{t-1}\) = hidden state from the previous time step
- \(h_t\) = updated hidden state at the current time step
- \(f\) = recurrent neural network (RNN) function

In [51]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

In [52]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

### Decoder with Additive Attention

The decoder is responsible for generating the translated sentence one token at a time. In this model, the decoder incorporates the **Bahdanau (Additive) Attention** mechanism, which enables it to selectively focus on different encoder outputs while predicting each target word.

At each decoding step:

- The previously generated or target word is converted into an embedding vector.
- The decoder's current hidden state is compared with all encoder hidden states using the additive attention mechanism.
- The resulting attention scores are normalized using the Softmax function to obtain attention weights.
- A context vector is created by computing the weighted sum of all encoder outputs.
- The context vector is combined with the current input embedding and passed through the decoder RNN.
- Finally, the decoder predicts the probability distribution of the next word in the target vocabulary.

This attention mechanism helps the decoder concentrate on the most relevant portions of the input sequence at every step, leading to more accurate translations. During training, **teacher forcing** is often applied, where the actual target word is used as the next input. During inference or evaluation, the decoder instead uses its own previously predicted word to continue generating the output sequence.

In [53]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

In [54]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

In [55]:
class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

## Training

### Data Preparation

Before training the attention-based Seq2Seq model, the dataset undergoes several preprocessing steps to ensure it is suitable for learning.

The preprocessing process includes:

- Loading the parallel French-English sentence dataset.
- Separating the dataset into input-output sentence pairs.
- Reversing the sentence pairs so that French serves as the input language and English as the target language.
- Converting all text into lowercase for consistency.
- Removing unnecessary punctuation and special characters.
- Tokenizing each sentence into individual words.
- Building separate vocabularies for both source and target languages.
- Assigning a unique integer index to every distinct word.
- Filtering out excessively long sentences to improve training efficiency.

The vocabulary mainly consists of:

- **word2index**: Maps each word to a unique numerical index.
- **index2word**: Converts numerical indices back into their corresponding words.
- **word2count**: Records how frequently each word appears in the dataset.

### Training Process

1. The encoder receives the complete input sentence and generates hidden states for every input token along with the final hidden state.
2. The decoder is initialized using the encoder's final hidden state.
3. At each decoding step, the Bahdanau attention mechanism calculates attention weights over all encoder hidden states.
4. A context vector is generated and combined with the decoder's input embedding before being passed to the decoder RNN.
5. The decoder predicts the next word in the target sequence.
6. The predicted output is compared with the corresponding target word to measure the prediction error.
7. The loss is calculated using **Negative Log-Likelihood (NLL) Loss**, as the decoder outputs log-softmax probabilities.
8. The model parameters, including those of the encoder, decoder, and attention layer, are updated through **Backpropagation Through Time (BPTT)** using an optimization algorithm such as **Adam**.

This training cycle is repeated for multiple epochs, enabling the model to gradually learn the relationship between French input sentences and their corresponding English translations.

In [56]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [57]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [58]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [59]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [60]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [61]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [62]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [ ]:
hidden_size = 128
batch_size = 32
EPOCHS = 20

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
encoder.train()
decoder.train()

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...


In [ ]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je suis veinard
= i m lucky
< i m in paris <EOS>

> t es assez bonne
= you re pretty good
< you re pretty good <EOS>

> elle est chanceuse
= she is lucky
< she is quite <EOS>

> vous etes vraiment splendides
= you re really gorgeous
< you re really gorgeous <EOS>

> vous etes courtoises
= you re courteous
< you re in luck <EOS>

> on m utilise
= i m being used
< i m just tired <EOS>

> vous etes imprudente
= you re foolish
< you re a traitor <EOS>

> il est toujours debout
= he is still standing
< he is still standing <EOS>

> nous avons toujours faim
= we re always hungry
< we re always hungry <EOS>

> tu es tres astucieuse
= you re very astute
< you re very astute <EOS>



## Discussion

In this lab, an attention-based Encoder-Decoder Seq2Seq model was successfully implemented for French-to-English machine translation. The encoder generated hidden representations for each input token, while the decoder utilized the Bahdanau attention mechanism to selectively focus on the most relevant encoder outputs during every decoding step. This approach overcomes the limitations of the traditional Seq2Seq model, which relies only on a single fixed-length context vector.

The experiment covered the complete machine translation pipeline, including dataset preprocessing, vocabulary generation, model training, and translation prediction. The results indicated that the model performs better on shorter and frequently occurring sentence patterns, whereas longer or less common sentences are comparatively more challenging to translate accurately. Furthermore, the translation performance is influenced by factors such as the size of the training dataset, the number of training epochs, the hidden layer dimensions, and the effectiveness of the attention mechanism.

## Conclusion

This lab provided practical experience in implementing an attention-based Seq2Seq model for neural machine translation. The Bahdanau attention mechanism enhanced the basic Encoder-Decoder architecture by enabling the decoder to utilize information from all encoder hidden states instead of relying solely on the final hidden state. As a result, the model is better able to capture relationships between source and target sequences, leading to improved translation performance. Overall, the experiment strengthened the understanding of attention mechanisms and their importance in modern sequence-to-sequence learning.